### **Import libraries**: 

In [7]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import sys
import fastparquet

### **Load Parquet file into a Pandas DataFrame object and calculate mean sentiment, sentiment standard deviation, number of articles, number of strongly negative articles, and sentiment momentum.**

In [8]:
def load_and_calculate(parquet_file_name_param):
    df = pd.read_parquet(parquet_file_name_param)
    tickers = list(set(df.index.get_level_values(0)))
    tickers.sort()
    dates = list(set(df.index.get_level_values(1)))
    dates = [str(element) for element in dates]
    dates.sort()
    date_today = str(datetime.today() - timedelta(days=2))[:10]
    iterables = [tickers, pd.to_datetime(dates)]
    index = pd.MultiIndex.from_product(iterables, names=["Ticker", "Timestamp"])
    df_aggregated = pd.DataFrame(index = index, columns = ["sent_mean", "sent_std", "news_count", "pct_strong_negative", "sent_momentum"])
    df_aggregated.sort_index(inplace = True, level = 1, ascending = False, sort_remaining = False)
    
    for ticker in tickers:
        for date in dates:
            try:
                sent_diff = df.loc[ticker, date]["pos"].sub(df.loc[ticker, date]["neg"], axis = 0)
                sent_mean_today = sent_diff.mean()
                sent_std_today = sent_diff.std()
                news_count_today = len(sent_diff)
                if news_count_today == 1:
                    sent_std_today = 0
                pct_strong_negative_today = (len(df.loc[ticker, date]["neg"][df.loc[ticker, date]["neg"]>0.7]) / len(df.loc[ticker, date]["neg"]))*100
                try:
                    sent_5_day_avg = np.mean([(df.loc[ticker, pd.Timestamp(date) - timedelta(days = i)]["pos"].sub(df.loc[ticker, pd.Timestamp(date) - timedelta(days = i)]["neg"], axis = 0)).mean() for i in range(5)])
                    sent_momentum = sent_mean_today - sent_5_day_avg
                    df_aggregated.loc[(ticker, date), df_aggregated.columns] = sent_mean_today, sent_std_today, news_count_today, pct_strong_negative_today, sent_momentum
                    
                except KeyError:
                    df_aggregated.loc[(ticker, date), df_aggregated.columns] = sent_mean_today, sent_std_today, news_count_today, pct_strong_negative_today, np.nan
                
            except KeyError:
                pass
                #print("No article was published about " + str(ticker) + " today.")


    df_aggregated[df_aggregated.columns] = df_aggregated[df_aggregated.columns].apply(pd.to_numeric, axis = 1)
    return df_aggregated
    #df_aggregated.index.set_levels(new_time_index level=1, inplace=True)


### **Main program used to collectively execute the whole script.**

In [9]:
def main():
    parquet_file_name = "daily_ticker_headlines_with_FinBERT_scoring_2.parquet"
    df_with_alt_data_features = load_and_calculate(parquet_file_name)
    print(df_with_alt_data_features)
    #df_with_alt_data_features.to_parquet("data_alt_features.parquet")
main()

/var/folders/90/y97m2k6n2z745vb16qxrb4cc0000gn/T/ipykernel_3376/1736018124.py:17: PerformanceWarning: indexing past lexsort depth may impact performance.
  sent_diff = df.loc[ticker, date]["pos"].sub(df.loc[ticker, date]["neg"], axis = 0)
/var/folders/90/y97m2k6n2z745vb16qxrb4cc0000gn/T/ipykernel_3376/1736018124.py:23: PerformanceWarning: indexing past lexsort depth may impact performance.
  pct_strong_negative_today = (len(df.loc[ticker, date]["neg"][df.loc[ticker, date]["neg"]>0.7]) / len(df.loc[ticker, date]["neg"]))*100
/var/folders/90/y97m2k6n2z745vb16qxrb4cc0000gn/T/ipykernel_3376/1736018124.py:25: PerformanceWarning: indexing past lexsort depth may impact performance.
  sent_5_day_avg = np.mean([(df.loc[ticker, pd.Timestamp(date) - timedelta(days = i)]["pos"].sub(df.loc[ticker, pd.Timestamp(date) - timedelta(days = i)]["neg"], axis = 0)).mean() for i in range(5)])


                   sent_mean  sent_std  news_count  pct_strong_negative  \
Ticker Timestamp                                                          
AAPL   2026-04-04   0.090954  0.233888         5.0             0.000000   
AMD    2026-04-04   0.449302  0.412799         3.0             0.000000   
AMZN   2026-04-04   0.084157  0.352984        13.0             0.000000   
BABA   2026-04-04  -0.018542  0.000000         1.0             0.000000   
BAC    2026-04-04  -0.256606  0.447565         2.0             0.000000   
...                      ...       ...         ...                  ...   
TSM    2026-03-31   0.066644  0.532387         7.0            14.285714   
UNH    2026-03-31        NaN       NaN         NaN                  NaN   
V      2026-03-31   0.094814  0.000000         1.0             0.000000   
WMT    2026-03-31        NaN       NaN         NaN                  NaN   
XOM    2026-03-31  -0.178109  0.778681         5.0            40.000000   

                   sent_